# 01 — Adaptive Query Execution (AQE) en acción

AQE es el secreto de por qué Spark 3.x es dramáticamente más rápido que
Spark 2.x para muchos casos. **Y viene prendido por default.**

Lo que hace:
1. **Coalesce post-shuffle**: si quedaron 200 particiones pero 150
   están casi vacías, las une.
2. **Skew join optimization**: detecta una partición gigante y la
   subdivide en runtime.
3. **Sort-merge → broadcast join**: si en runtime la tabla "chica" entró
   en memoria, la convierte automáticamente.

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("04_01_aqe")
    .master("spark://spark-master:7077")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.adaptive.skewJoin.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("AQE enabled:", spark.conf.get("spark.sql.adaptive.enabled"))

In [ ]:
df = spark.read.parquet("/home/jovyan/work/data/sample/ventas/parquet_limpio")
print("Filas:", df.count())
print("Particiones:", df.rdd.getNumPartitions())

## Demo: shuffle partitions adaptativas

Forzamos 200 particiones post-shuffle, pero los datos son chicos.
AQE debería coalescer en runtime.

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", "200")

import time
t = time.time()
result = (
    df.groupBy("ciudad", "producto")
      .agg(F.count("*").alias("n"), F.avg("precio"))
      .collect()
)
print(f"Con AQE on: {time.time() - t:.2f}s, {len(result)} filas")

# Ahora sin AQE
spark.conf.set("spark.sql.adaptive.enabled", "false")
t = time.time()
result = (
    df.groupBy("ciudad", "producto")
      .agg(F.count("*").alias("n"), F.avg("precio"))
      .collect()
)
print(f"Con AQE off: {time.time() - t:.2f}s, {len(result)} filas")

spark.conf.set("spark.sql.adaptive.enabled", "true")

## Cómo verificar en la Spark UI

1. Andá a la pestaña **SQL** del job.
2. Buscá el nodo `Exchange` (el shuffle).
3. Si AQE actuó, vas a ver `CoalescedPartitions` o `OptimizeSkewedJoin`
   justo después.
4. El número final de partitions va a ser menor que
   `spark.sql.shuffle.partitions`.

**Si AQE está apagado, prendelo YA.** Es la mejora de performance más
grande con cero costo.

## ✅ Checkpoint

- [x] Sé qué hace AQE.
- [x] Sé verificar en la UI que actuó.
- [x] Sé cuándo AQE no alcanza (config mal seteada, datos muy skewed).

**Siguiente**: `02_shuffle_partitions.ipynb` — cómo elegir el número
correcto de particiones.